# PyTorch MNIST CNN demo
Runtime > Change runtime type > GPU (optional).

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(dev)

cuda


In [2]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train = DataLoader(datasets.MNIST('data', train=True, download=True, transform=tf), batch_size=128, shuffle=True)
test = DataLoader(datasets.MNIST('data', train=False, transform=tf), batch_size=1000)


  0%|          | 0.00/9.91M [00:00<?, ?B/s]
  1%|          | 98.3k/9.91M [00:00<00:12, 761kB/s]
  4%|▍         | 393k/9.91M [00:00<00:05, 1.65MB/s]
 16%|█▌        | 1.61M/9.91M [00:00<00:01, 5.13MB/s]
 66%|██████▌   | 6.55M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]

  0%|          | 0.00/28.9k [00:00<?, ?B/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 457kB/s]

  0%|          | 0.00/1.65M [00:00<?, ?B/s]
  6%|▌         | 98.3k/1.65M [00:00<00:02, 743kB/s]
 24%|██▍       | 393k/1.65M [00:00<00:00, 1.61MB/s]
 97%|█████████▋| 1.61M/1.65M [00:00<00:00, 5.02MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.11MB/s]

  0%|          | 0.00/4.54k [00:00<?, ?B/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.02MB/s]



In [3]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3, 1)
        self.c2 = nn.Conv2d(32, 64, 3, 1)
        self.d = nn.Dropout(0.25)
        self.f1 = nn.Linear(9216, 128)
        self.f2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.c1(x))
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        x = self.d(torch.flatten(x, 1))
        return self.f2(F.relu(self.f1(x)))

model = CNN().to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [4]:
for epoch in range(3):
    model.train()
    for x, y in train:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        F.cross_entropy(model(x), y).backward()
        opt.step()
    model.eval(); ok = 0
    with torch.no_grad():
        for x, y in test:
            ok += (model(x.to(dev)).argmax(1).cpu() == y).sum().item()
    print(f'epoch {epoch+1}: test acc {ok/len(test.dataset):.4f}')

epoch 1: test acc 0.9851
epoch 2: test acc 0.9882
epoch 3: test acc 0.9871
